<a href="https://colab.research.google.com/github/dilnazaabay-hub/heist-squad-bot/blob/main/telegrambot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytelegrambotapi

In [ ]:
MY_TOKEN = "8680340689:AAEIhclt0TFmudjFud2wL6NkK-WctZIA6UU"

In [ ]:
import telebot
from telebot import types
import random

bot = telebot.TeleBot(MY_TOKEN)

user_games = {}
leaderboard = {}

In [ ]:
CHARACTERS = [
    {"name": "Nova", "role": "Hacker", "rarity": "Epic", "hack": 9, "fight": 3, "stealth": 6, "strategy": 7},
    {"name": "Rex", "role": "Fighter", "rarity": "Rare", "hack": 2, "fight": 9, "stealth": 4, "strategy": 5},
    {"name": "Mira", "role": "Medic", "rarity": "Common", "hack": 3, "fight": 4, "stealth": 5, "strategy": 8},
    {"name": "Ghost", "role": "Spy", "rarity": "Epic", "hack": 6, "fight": 4, "stealth": 10, "strategy": 6},
    {"name": "Leo", "role": "Leader", "rarity": "Rare", "hack": 4, "fight": 6, "stealth": 5, "strategy": 9},
    {"name": "Byte", "role": "Hacker", "rarity": "Common", "hack": 7, "fight": 2, "stealth": 5, "strategy": 5},
    {"name": "Tank", "role": "Fighter", "rarity": "Common", "hack": 1, "fight": 10, "stealth": 2, "strategy": 4},
    {"name": "Luna", "role": "Spy", "rarity": "Rare", "hack": 5, "fight": 3, "stealth": 9, "strategy": 7},
    {"name": "Atlas", "role": "Leader", "rarity": "Epic", "hack": 5, "fight": 7, "stealth": 4, "strategy": 10},
    {"name": "Ivy", "role": "Medic", "rarity": "Rare", "hack": 4, "fight": 3, "stealth": 6, "strategy": 9}
]

In [ ]:
MISSIONS = {
    "easy": {
        "name": "School Data Rescue",
        "required": {"hack": 12, "fight": 10, "stealth": 10, "strategy": 12},
        "min_score": 45
    },
    "medium": {
        "name": "Museum Heist",
        "required": {"hack": 18, "fight": 15, "stealth": 20, "strategy": 18},
        "min_score": 65
    },
    "hard": {
        "name": "Cyber Bank Mission",
        "required": {"hack": 24, "fight": 20, "stealth": 24, "strategy": 23},
        "min_score": 85
    }
}

In [ ]:
def main_menu():
    markup = types.InlineKeyboardMarkup(row_width=1)
    markup.add(
        types.InlineKeyboardButton("🎮 Start new mission", callback_data="new_game"),
        types.InlineKeyboardButton("🏆 Leaderboard", callback_data="leaderboard"),
        types.InlineKeyboardButton("ℹ️ Help", callback_data="help")
    )
    return markup


def difficulty_menu():
    markup = types.InlineKeyboardMarkup(row_width=3)
    markup.add(
        types.InlineKeyboardButton("Easy", callback_data="level_easy"),
        types.InlineKeyboardButton("Medium", callback_data="level_medium"),
        types.InlineKeyboardButton("Hard", callback_data="level_hard")
    )
    return markup

In [ ]:
def character_power(character):
    return (
        character["hack"]
        + character["fight"]
        + character["stealth"]
        + character["strategy"]
    )


def show_character(character, number):
    return (
        f"{number}. {character['name']} | {character['role']} | {character['rarity']}\n"
        f"Hack: {character['hack']} | Fight: {character['fight']} | "
        f"Stealth: {character['stealth']} | Strategy: {character['strategy']}\n\n"
    )


def build_character_keyboard(characters, selected):
    markup = types.InlineKeyboardMarkup(row_width=2)

    for i, character in enumerate(characters):
        if i in selected:
            text = f"✅ {i + 1}. {character['name']}"
        else:
            text = f"{i + 1}. {character['name']}"

        markup.add(types.InlineKeyboardButton(text, callback_data=f"pick_{i}"))

    markup.add(
        types.InlineKeyboardButton("🚀 Start mission", callback_data="finish_team"),
        types.InlineKeyboardButton("🔄 Reset team", callback_data="reset_team")
    )

    return markup

In [ ]:
def create_game(chat_id, difficulty):
    characters = random.sample(CHARACTERS, 8)

    user_games[chat_id] = {
        "difficulty": difficulty,
        "characters": characters,
        "selected": []
    }

    mission = MISSIONS[difficulty]

    text = f"🎯 Mission: {mission['name']}\n\n"

    text += "Mission requirements:\n"
    text += f"Hack needed: {mission['required']['hack']}\n"
    text += f"Fight needed: {mission['required']['fight']}\n"
    text += f"Stealth needed: {mission['required']['stealth']}\n"
    text += f"Strategy needed: {mission['required']['strategy']}\n\n"

    text += "Scoring rules:\n"
    text += "- Too low skill = +8 points\n"
    text += "- Balanced skill = +20 points\n"
    text += "- Overpowered skill = +12 points\n"
    text += "- Balanced team composition = +15 points\n"
    text += "- Unbalanced team composition = -10 points\n"
    text += "- Random event can add or remove points\n\n"

    text += "How to win:\n"
    text += "1. Choose exactly 4 characters.\n"
    text += "2. Team should have at least 3 different roles.\n"
    text += "3. Try to reach requirements, but avoid being too overpowered.\n\n"

    text += "Available characters:\n\n"

    for i, character in enumerate(characters):
        text += show_character(character, i + 1)

    bot.send_message(
        chat_id,
        text,
        reply_markup=build_character_keyboard(characters, [])
    )

In [ ]:
def calculate_team_stats(team):
    stats = {
        "hack": 0,
        "fight": 0,
        "stealth": 0,
        "strategy": 0
    }

    for character in team:
        stats["hack"] += character["hack"]
        stats["fight"] += character["fight"]
        stats["stealth"] += character["stealth"]
        stats["strategy"] += character["strategy"]

    return stats


def check_balance(team):
    powers = [character_power(character) for character in team]
    average_power = sum(powers) / len(powers)

    roles = [character["role"] for character in team]
    unique_roles = set(roles)

    if average_power < 18:
        return False, "Team is too weak."

    if average_power > 30:
        return False, "Team is too overpowered."

    if len(unique_roles) < 3:
        return False, "Team needs at least 3 different roles."

    return True, "Balanced team composition detected."

In [ ]:
def calculate_skill_points(skill_value, required_value):
    if skill_value < required_value:
        return 8, "not enough"

    if skill_value <= required_value + 8:
        return 20, "balanced"

    return 12, "overpowered"

In [ ]:
def evaluate_mission(chat_id):
    game = user_games[chat_id]
    mission = MISSIONS[game["difficulty"]]

    selected_indexes = game["selected"]
    team = [game["characters"][i] for i in selected_indexes]

    is_balanced, balance_message = check_balance(team)
    stats = calculate_team_stats(team)

    score = 0
    explanation = ""

    for skill, required_value in mission["required"].items():
        points, status = calculate_skill_points(stats[skill], required_value)
        score += points

        if status == "not enough":
            explanation += (
                f"⚠️ {skill}: not enough "
                f"({stats[skill]}/{required_value}) = +{points}\n"
            )

        elif status == "balanced":
            explanation += (
                f"✅ {skill}: balanced "
                f"({stats[skill]}/{required_value}) = +{points}\n"
            )

        else:
            explanation += (
                f"🔥 {skill}: overpowered "
                f"({stats[skill]}/{required_value}) = +{points}\n"
            )

    if is_balanced:
        score += 15
        balance_points = "+15"
    else:
        score -= 10
        balance_points = "-10"

    random_event = random.choice([
        ("Security system was weak.", 10),
        ("Unexpected guard appeared.", -8),
        ("Team found a secret shortcut.", 7),
        ("Communication problem happened.", -5)
    ])

    score += random_event[1]

    if score >= mission["min_score"]:
        result = "🎉 SUCCESS"
    elif score >= mission["min_score"] - 15:
        result = "😐 PARTIAL SUCCESS"
    else:
        result = "💥 FAILED"

    if chat_id not in leaderboard or score > leaderboard[chat_id]:
        leaderboard[chat_id] = score

    reply = f"{result}\n\n"
    reply += f"Mission: {mission['name']}\n"
    reply += f"Final score: {score}\n"
    reply += f"Score needed for success: {mission['min_score']}\n\n"

    reply += "Your team:\n"
    for character in team:
        reply += f"- {character['name']} ({character['role']})\n"

    reply += "\nYour team stats:\n"
    reply += f"Hack: {stats['hack']} / needed {mission['required']['hack']}\n"
    reply += f"Fight: {stats['fight']} / needed {mission['required']['fight']}\n"
    reply += f"Stealth: {stats['stealth']} / needed {mission['required']['stealth']}\n"
    reply += f"Strategy: {stats['strategy']} / needed {mission['required']['strategy']}\n\n"

    reply += f"Team balance result: {balance_message} ({balance_points})\n"
    reply += f"Random event: {random_event[0]} ({random_event[1]} points)\n\n"

    reply += "Score explanation:\n"
    reply += explanation

    return reply